In [ ]:
import os
import sys
import re

import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import igraph as ig
import leidenalg as la
import celloracle as co
from pathlib import Path

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [ ]:
import celloracle as co
co.__version__


'0.10.12'

In [ ]:
OUTDIR = Path("GRN_mouse_outputs")
OUTDIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "NKT":  "GRN_lineage_NKT.h5ad",
    "MAIT": "GRN_lineage_MAIT.h5ad",
    "gdt":  "GRN_lineage_γδT.h5ad",
}

CLUSTER_COL    = "paper_subtype_strict"
MAIT_STAGE_COL = "paper_stage"


In [ ]:
def prepare_for_oracle(adata):
    if "gene_symbols" not in adata.var:
        adata.var["gene_symbols"] = adata.var_names

    if "counts" not in adata.layers:
        adata.layers["counts"] = adata.X.copy()

    if "X_pca" not in adata.obsm:
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        sc.pp.pca(adata, n_comps=50)
        print("[prepare_for_oracle] PCA computed -> obsm['X_pca']")
    return adata


adata_nkt  = prepare_for_oracle(sc.read_h5ad(FILES["NKT"]))
adata_mait = prepare_for_oracle(sc.read_h5ad(FILES["MAIT"]))
adata_gdt  = prepare_for_oracle(sc.read_h5ad(FILES["gdt"]))

print("preparation ready")


[prepare_for_oracle] PCA computed -> obsm['X_pca']
[prepare_for_oracle] PCA computed -> obsm['X_pca']
preparation ready


In [ ]:
base_GRN = co.data.load_mouse_promoter_base_GRN(version='mm10_gimmemotifsv5_fpr2')
display(base_GRN.head(), base_GRN.shape)

Loading prebuilt promoter base-GRN. Version: mm10_gimmemotifsv5_fpr2


,peak_id,gene_short_name,9430076c15rik,Ac002126.6,Ac012531.1,Ac226150.2,Afp,Ahctf1,Ahr,Ahrr,...,Znf784,Znf8,Znf816,Znf85,Zscan10,Zscan16,Zscan22,Zscan26,Zscan31,Zscan4
0,chr10_100014821_100015921,Kitl,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,chr10_100334676_100335776,Gm4301,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,chr10_100339838_100340938,Gm4303,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,chr10_100344970_100346070,Gm4302,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,chr10_100350091_100351191,Gm4302,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


(29308, 1096)

In [ ]:
def safe_tag(s: str) -> str:
    return (s.replace("γ","g").replace("δ","d").replace("/","_").replace(" ","_"))

def pick_first_present_label(present_labels, candidates):
    for c in candidates:
        if c in present_labels:
            return c
    return None

NKT_CANDIDATES = ["NKTp", "NKTP", "NKTp "]
GDT_CANDIDATES = ["Tγδp", "γδTp", "gdTp", "TgdP", "gammadeltaP", "Tgd_p"]

present_nkt = set(adata_nkt.obs[CLUSTER_COL].astype(str).unique())
nktp = pick_first_present_label(present_nkt, NKT_CANDIDATES) or "NKTp"
print("NKT control chosen:", nktp)

adata_mait_ctrl = adata_mait[adata_mait.obs[MAIT_STAGE_COL].astype(str) == "S2"].copy()

adata_mait_ctrl.obs[CLUSTER_COL] = "MAIT_S2"

present_gdt = set(adata_gdt.obs[CLUSTER_COL].astype(str).unique())
gdtp = pick_first_present_label(present_gdt, GDT_CANDIDATES) or "Tγδp"
print("γδT control chosen:", gdtp)


NKT control chosen: NKTp
γδT control chosen: Tγδp


In [ ]:
print("MAIT control chosen: MAIT_S2, n_cells =", adata_mait_ctrl.n_obs)

MAIT control chosen: MAIT_S2, n_cells = 799


In [ ]:
def run_grn_pipeline_one_control(
    adata_full,
    control_label: str,
    cluster_col: str,
    base_GRN,
    outdir: Path,
    n_hvg: int = 3000,
    alpha: int = 10,
    n_jobs: int = -1
):
    outdir.mkdir(parents=True, exist_ok=True)

    ad = adata_full[adata_full.obs[cluster_col].astype(str) == control_label].copy()
    assert ad.n_obs > 0, f"No cells for {control_label}"

    ad_tmp = ad.copy()
    sc.pp.normalize_total(ad_tmp, target_sum=1e4)
    sc.pp.log1p(ad_tmp)
    sc.pp.highly_variable_genes(ad_tmp, n_top_genes=n_hvg, flavor="seurat_v3")
    hvg = ad_tmp.var.index[ad_tmp.var["highly_variable"]].tolist()
    if len(hvg) < 1000:
        hvg = ad.var_names.tolist()
    ad = ad[:, ad.var_names.isin(hvg)].copy()

    if "counts" not in ad.layers:
        ad.layers["counts"] = ad.X.copy()
    ad.X = ad.layers["counts"].copy()

    if "X_pca" not in ad.obsm:
        sc.pp.normalize_total(ad, target_sum=1e4)
        sc.pp.log1p(ad)
        sc.pp.pca(ad, n_comps=50)

    oracle = co.Oracle()
    oracle.import_anndata_as_raw_count(
        adata=ad, cluster_column_name=cluster_col, embedding_name="X_pca"
    )
    oracle.import_TF_data(TF_info_matrix=base_GRN)

    oracle.perform_PCA()
    n_cell = oracle.adata.shape[0]
    n_comps = min(30, 50, oracle.pca.components_.shape[0])
    k = max(10, int(0.025 * n_cell))
    oracle.knn_imputation(n_pca_dims=n_comps, k=k, balanced=True,
                          b_sight=k*8, b_maxl=k*4, n_jobs=n_jobs)

    links = oracle.get_links(
        cluster_name_for_GRN_unit=cluster_col,
        alpha=alpha, verbose_level=10, n_jobs=n_jobs
    )
    df = links.links_dict[control_label].copy()


    fname = f"raw_GRN_for_{safe_tag(control_label)}.tsv"
    outpath = outdir / fname
    df.to_csv(outpath, sep="\t", index=False)
    print(f"[{control_label}] saved:", outpath)
    return df, outpath


raw_paths = []

# NKT
df_nkt, path_nkt = run_grn_pipeline_one_control(
    adata_full=adata_nkt,
    control_label=nktp,
    cluster_col=CLUSTER_COL,
    base_GRN=base_GRN,
    outdir=OUTDIR / "NKT",
    n_hvg=3000, alpha=10, n_jobs=-1
)
raw_paths.append(path_nkt)

# MAIT
df_mait, path_mait = run_grn_pipeline_one_control(
    adata_full=adata_mait_ctrl,
    control_label="MAIT_S2",
    cluster_col=CLUSTER_COL,
    base_GRN=base_GRN,
    outdir=OUTDIR / "MAIT",
    n_hvg=3000, alpha=10, n_jobs=-1
)
raw_paths.append(path_mait)

# γδT
df_gdt, path_gdt = run_grn_pipeline_one_control(
    adata_full=adata_gdt,
    control_label=gdtp,
    cluster_col=CLUSTER_COL,
    base_GRN=base_GRN,
    outdir=OUTDIR / "gdt",
    n_hvg=3000, alpha=10, n_jobs=-1
)
raw_paths.append(path_gdt)

raw_paths


  0%|          | 0/1 [00:00<?, ?it/s]

Inferring GRN for NKTp...


  0%|          | 0/2302 [00:00<?, ?it/s]

[NKTp] saved: GRN_mouse_outputs/NKT/raw_GRN_for_NKTp.tsv


  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

Inferring GRN for Tγδp...


  0%|          | 0/2223 [00:00<?, ?it/s]

[Tγδp] saved: GRN_mouse_outputs/gdt/raw_GRN_for_Tgdp.tsv


[PosixPath('GRN_mouse_outputs/NKT/raw_GRN_for_NKTp.tsv'),
 PosixPath('GRN_mouse_outputs/MAIT/raw_GRN_for_MAIT_S2.tsv'),
 PosixPath('GRN_mouse_outputs/gdt/raw_GRN_for_Tgdp.tsv')]

In [ ]:
dfs = []
for p in raw_paths:
    df = pd.read_csv(p, sep="\t")
    tag = Path(p).stem.replace("raw_GRN_for_", "")
    df["cluster"] = tag
    dfs.append(df)

raw_all = pd.concat(dfs, axis=0, ignore_index=True)
RAW_ALL_PATH = OUTDIR / "raw_GRN_combined.tsv"
raw_all.to_csv(RAW_ALL_PATH, sep="\t", index=False)
print("Combined GRN saved:", RAW_ALL_PATH, "shape:", raw_all.shape)


Combined GRN saved: GRN_mouse_outputs/raw_GRN_combined.tsv shape: (181754, 7)


In [ ]:
TF_DIR = OUTDIR / "TF_Graphs"
TF_DIR.mkdir(parents=True, exist_ok=True)

print("columns used:", SRC_COL, TGT_COL, W_COL)
print("TF→TF edges:", raw_tf_tf.shape)

percent_steps = list(range(100, 0, -10))  # 100, 90, ..., 10

def threshold_by_percentile(df, col, pct):
    if df.empty or pct >= 100:
        return df.copy()
    thr = np.percentile(df[col].values, pct)
    return df[df[col] >= thr].copy()

def cluster_tf_graph(edges_df, src_col, tgt_col, weight_col):
    nodes = sorted(set(edges_df[src_col]) | set(edges_df[tgt_col]))
    idx = {n:i for i,n in enumerate(nodes)}
    g = ig.Graph(
        n=len(nodes),
        edges=[(idx[s], idx[t]) for s,t in edges_df[[src_col, tgt_col]].itertuples(index=False)],
        directed=True
    )
    if weight_col in edges_df.columns:
        g.es["weight"] = edges_df[weight_col].values
    ug = g.as_undirected(combine_edges="sum")
    part = la.find_partition(
        ug, la.RBConfigurationVertexPartition,
        weights=ug.es["weight"] if "weight" in ug.es.attributes() else None,
        resolution_parameter=1.0
    )
    labels = np.empty(len(nodes), dtype=int)
    for cid, verts in enumerate(part):
        labels[list(verts)] = cid
    return pd.DataFrame({"TF": nodes, "community": labels})

summary = []
for pct in percent_steps:
    step_df = threshold_by_percentile(raw_tf_tf, W_COL, pct)
    if step_df.empty:
        print(f"{pct}% -> empty"); continue

    tf_comm = cluster_tf_graph(step_df, SRC_COL, TGT_COL, W_COL)
    tf_comm_path = TF_DIR / f"tf_communities_step_{pct}pct.csv"
    edges_path   = TF_DIR / f"tf_community_edges_step_{pct}pct.csv"
    tf_comm.to_csv(tf_comm_path, index=False)
    step_df.to_csv(edges_path, index=False)

    summary.append({
        "step_pct": pct,
        "n_nodes": tf_comm.shape[0],
        "n_edges": step_df.shape[0],
        "n_communities": tf_comm["community"].nunique()
    })
    print(f"{pct}% -> nodes={tf_comm.shape[0]} edges={step_df.shape[0]} comm={tf_comm['community'].nunique()}")

summary_df = pd.DataFrame(summary).sort_values("step_pct", ascending=False)
summary_df.to_csv(TF_DIR / "tf_tf_grn_summary.txt", sep="\t", index=False)
summary_df


columns used: source target coef_abs
TF→TF edges: (11122, 7)
100% -> nodes=207 edges=11122 comm=5
90% -> nodes=138 edges=1113 comm=7
80% -> nodes=171 edges=2225 comm=6
70% -> nodes=186 edges=3337 comm=6
60% -> nodes=191 edges=4449 comm=6
50% -> nodes=197 edges=5561 comm=6
40% -> nodes=202 edges=6673 comm=6
30% -> nodes=206 edges=7785 comm=5
20% -> nodes=206 edges=8897 comm=6
10% -> nodes=207 edges=10009 comm=5


,step_pct,n_nodes,n_edges,n_communities
0,100,207,11122,5
1,90,138,1113,7
2,80,171,2225,6
3,70,186,3337,6
4,60,191,4449,6
5,50,197,5561,6
6,40,202,6673,6
7,30,206,7785,5
8,20,206,8897,6
9,10,207,10009,5


In [ ]:
tf10 = pd.read_csv(TF_DIR / "tf_communities_step_10pct.csv")
tf10["TF"].dropna().drop_duplicates().to_csv(TF_DIR / "tf_list.txt", index=False, header=False)
print("saved:", TF_DIR / "tf_list.txt")


saved: GRN_mouse_outputs/TF_Graphs/tf_list.txt


In [ ]:
TF_DIR = OUTDIR / "TF_Graphs"
figdir = TF_DIR / "figures"
figdir.mkdir(parents=True, exist_ok=True)

tf10 = pd.read_csv(TF_DIR / "tf_communities_step_10pct.csv")
edges10 = pd.read_csv(TF_DIR / "tf_community_edges_step_10pct.csv")
print(tf10.shape, edges10.shape)


(207, 2) (10009, 7)


In [ ]:
comm_counts = tf10["community"].value_counts().sort_index()
comm_counts.to_csv(TF_DIR / "community_sizes_10pct.tsv", sep="\t")
plt.figure(figsize=(10,4))
plt.bar(comm_counts.index.astype(str), comm_counts.values)
plt.xlabel("Community"); plt.ylabel("# TFs"); plt.title("Community sizes (10%)")
plt.show()


In [ ]:
deg = pd.Series(edges10[SRC_COL].value_counts(), name="outdeg").to_frame()
deg["indeg"] = edges10[TGT_COL].value_counts()
deg = deg.fillna(0)
deg["deg"] = deg["outdeg"] + deg["indeg"]
deg = deg.reset_index().rename(columns={"index":"TF"})
deg = deg.merge(tf10, on="TF", how="left")
deg_sorted = deg.sort_values(["community","deg"], ascending=[True, False])
deg_sorted.to_csv(TF_DIR / "top_TFs_by_degree_10pct.tsv", sep="\t", index=False)

topk = []
for c in sorted(tf10["community"].unique()):
    sub = deg_sorted[deg_sorted["community"]==c].head(10)
    topk.append(sub.assign(rank=range(1, len(sub)+1)))
topk = pd.concat(topk, axis=0, ignore_index=True)
topk[["community","rank","TF","deg"]]

,community,rank,TF,deg
0,0,1,Klf2,381.0
1,0,2,Klf3,377.0
2,0,3,Gata3,199.0
3,0,4,Klf9,190.0
4,0,5,Arntl2,182.0
5,0,6,Id3,179.0
6,0,7,Id2,163.0
7,0,8,Klf4,158.0
8,0,9,Rorc,148.0
9,0,10,Eno1,142.0


In [ ]:
tf2comm = dict(zip(tf10["TF"], tf10["community"]))

edges10 = edges10.copy()
edges10["src_c"] = edges10[SRC_COL].map(tf2comm)
edges10["tgt_c"] = edges10[TGT_COL].map(tf2comm)

ecomm = edges10.dropna(subset=["src_c","tgt_c"]).copy()
ecomm["src_c"] = ecomm["src_c"].astype(int)
ecomm["tgt_c"] = ecomm["tgt_c"].astype(int)

comms = sorted(tf10["community"].unique())
idx = {c:i for i,c in enumerate(comms)}
M = np.zeros((len(comms), len(comms)), dtype=float)

w = ecomm[W_COL].values if W_COL in ecomm.columns else np.ones(len(ecomm), dtype=float)
for s, t, ww in zip(ecomm["src_c"], ecomm["tgt_c"], w):
    M[idx[s], idx[t]] += ww

M_norm = (M / M.max()) if M.max() > 0 else M

plt.figure(figsize=(6,5))
plt.imshow(M_norm, aspect="auto")
plt.colorbar(label="normalized sum of weights")
plt.xticks(range(len(comms)), comms)
plt.yticks(range(len(comms)), comms)
plt.xlabel("target community"); plt.ylabel("source community")
plt.title("Community-to-community connectivity (10%)")
plt.tight_layout(); plt.savefig(figdir / "community_connectivity_10pct.png", dpi=200)
plt.show()


pd.DataFrame(M, index=comms, columns=comms).to_csv(TF_DIR / "community_connectivity_weights_10pct.tsv", sep="\t")
pd.DataFrame(M_norm, index=comms, columns=comms).to_csv(TF_DIR / "community_connectivity_weights_10pct_normalized.tsv", sep="\t")
